In [ ]:
import sys
import os
from pathlib import Path
import streamlit as st
import yt_dlp

root_dir = Path(__file__).resolve().parent.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

LOCAL_MODEL_PATH = (root_dir / "Models" / "qwen_local_model").resolve().as_posix()

try:
    from Tools import Agent_Widgets_Offline, Agent_Widgets_Online
except ModuleNotFoundError as e:
    st.error(f"Failed to Import the Packages / Modules")
    raise e

def get_pipeline():
    from transformers import pipeline
    return pipeline

instructions = (
    "You are an expert Content Strategy Agent and Video Analyst. Analyze the provided video "
    "context data completely. Respond ONLY with a comprehensive, well-structured article/blog post breaking down "
    "the core message. You MUST include a formal 'Table of Contents' at the very top, followed by "
    "detailed sectional breakdowns using clear headings, and a final summary synthesis. "
    "Do not include conversational filler or generic introduction text."
)

st.set_page_config(page_title="Smart Video Summarizer & Blogger", layout="centered")

st.title("Smart Video Summarizer & Blogger")

tab_online, tab_offline = st.tabs(["Online Summarizer", "Offline Summarizer"])

# Online

with tab_online:
    st.subheader("Online Video Summarizer & Blogger")
    
    api_key = st.text_input(
        "Enter your Gemini Api Key for Verification", 
        type="password", 
        help="Your Gemini Api Key", placeholder="Your Gemini Api Key",
        key="api_key_online"
    )
    
    video_source_online = st.radio("Select Video Source", ["Upload Local Video", "Enter YouTube Link"], key="source_online")
    
    uploaded_file_online = None
    youtube_url_online = ""
    
    if video_source_online == "Upload Local Video":
        uploaded_file_online = st.file_uploader("Upload a Local Video File", type=["mp4", "mov", "avi"], key="local_video_online")
    else:
        youtube_url_online = st.text_input("Paste YouTube Video Url", placeholder="Youtube Video Link", key="yt_url_online")

    if st.button("Generate Blog Summary", type="primary", key="btn_online"):
        if not api_key.strip():
            st.error("Authentication Failed! Please Provide a Valid Gemini Api Key for Verification")
        elif video_source_online == "Upload Local MP4 Video" and uploaded_file_online is None:
            st.warning("Please Upload a Valid Video File to Proceed")
        elif video_source_online == "Enter YouTube Link" and not youtube_url_online.strip():
            st.warning("Please Provide a Valid YouTube Address")
        else:
            with st.spinner("Gemini is Analyzing Video .."):
                try:
                    Agent_Widgets_Online.client.aio.api_key = api_key
                    Agent_Widgets_Online.client.models.api_key = api_key
                    
                    temp_path = Path("temp_online_video.mp4")
                    
                    if video_source_online == "Upload Local Video":
                        with open(temp_path, "wb") as f:
                            f.write(uploaded_file_online.getbuffer())
                    else:
                        ydl_opts = {
                            'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
                            'outtmpl': str(temp_path), 'quiet': True, 'no_warnings': True
                        }
                        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                            ydl.download([youtube_url_online.strip()])
                    
                    video_payload = Agent_Widgets_Online.client.files.upload(file=temp_path)
                    
                    response = Agent_Widgets_Online.client.models.generate_content(
                        model='gemini-2.5-flash',
                        contents=[video_payload, instructions]
                    )
                    blog_output = response.text
                    
                    if temp_path.exists():
                        os.remove(temp_path)
                    
                    if "API_KEY_INVALID" in blog_output or "invalid" in blog_output.lower() or "400" in blog_output:
                        st.warning("Your Gemini Api Key is Invalid, Please Try Again")
                    else:
                        st.session_state['online_video_blog_text'] = blog_output
                        if 'online_error' in st.session_state:
                            del st.session_state['online_error']
                        
                except Exception as e:
                    error_msg = str(e).lower()
                    if "api_key_invalid" in error_msg or "invalid key" in error_msg or "unauthorized" in error_msg:
                        st.session_state['online_error'] = "invalid_key"
                    else:
                        st.error(f"Failed to Produce Video Blog Summary")

    if st.session_state.get('online_error') == "invalid_key":
        st.warning("Your Gemini Api Key is Invalid, Please Try Again")

    if 'online_video_blog_text' in st.session_state:
        st.markdown("---")
        st.markdown(st.session_state['online_video_blog_text'])
        st.markdown("---")
        
        custom_name_online = st.text_input("Enter the Name of your Export File", placeholder="Export File Name", key="name_file_online")
        base_name_on = custom_name_online.strip() if custom_name_online.strip() else "online_video_summary"
        clean_filename_online = f"{base_name_on}.txt"
        
        st.download_button(
            label="Export your Blog Summary",
            data=st.session_state['online_video_blog_text'],
            file_name=clean_filename_online,
            mime="text/plain",
            use_container_width=True, key="download_btn_online", on_click="ignore"
        )

# Offline

with tab_offline:
    st.subheader("Offline Video Summarizer & Blogger")

    @st.cache_resource
    def load_offline_pipelines():
        pipeline_loader = get_pipeline()
        qwen_pipeline = pipeline_loader("text-generation", model=LOCAL_MODEL_PATH)
        whisper_pipeline = pipeline_loader("automatic-speech-recognition", model="openai/whisper-tiny.en")
        return qwen_pipeline, whisper_pipeline

    try:
        local_qwen, local_whisper = load_offline_pipelines()
        model_ready = True
    except Exception as model_err:
        st.error(f"Unable to Load Model")
        model_ready = False

    video_source_offline = st.radio("Select Video Source", ["Upload Local Video", "Enter YouTube Link"], key="source_offline")
    
    uploaded_file_offline = None
    youtube_url_offline = ""
    
    if video_source_offline == "Upload Local Video":
        uploaded_file_offline = st.file_uploader("Upload a Local Video File", type=["mp4", "mov", "avi"], key="local_video_offline")
    else:
        youtube_url_offline = st.text_input("Paste YouTube Video Url", placeholder="Youtube Video Link", key="yt_url_offline")

    if st.button("Generate Blog Summary", disabled=not model_ready, key="btn_offline"):
        if video_source_offline == "Upload Local MP4 Video" and uploaded_file_offline is None:
            st.warning("Please Upload a Valid Video File to Proceed")
        elif video_source_offline == "Enter YouTube Link" and not youtube_url_offline.strip():
            st.warning("Please Provide a Valid YouTube Address")
        else:
            with st.spinner("Model is Analyzing Video ..."):
                try:
                    temp_path_off = Path("temp_offline_video.mp4")
                    
                    if video_source_offline == "Upload Local MP4 Video":
                        with open(temp_path_off, "wb") as f:
                            f.write(uploaded_file_offline.getbuffer())
                    else:
                        ydl_opts_off = {
                            'format': 'bestaudio/best',
                            'outtmpl': str(temp_path_off), 'quiet': True, 'no_warnings': True
                        }
                        with yt_dlp.YoutubeDL(ydl_opts_off) as ydl:
                            ydl.download([youtube_url_offline.strip()])
                    
                    transcription_data = local_whisper(str(temp_path_off), return_timestamps=True)
                    extracted_text_transcript = transcription_data["text"]
                    
                    if temp_path_off.exists():
                        os.remove(temp_path_off)
                        
                    with st.spinner("Model is Compiling Blog Summary ..."):
                        user_prompt = f"Video audio data text transcript:\n{extracted_text_transcript}"
                        
                        recipe_output = Agent_Widgets_Offline.Text(
                            prompt=user_prompt,
                            instruction=instructions,
                            generator_pipeline=local_qwen
                        )
                        
                        st.session_state['offline_video_blog_text'] = recipe_output
                        
                except Exception as e:
                    st.error(f"Failed to Produce Video Blog Summary")

    if 'offline_video_blog_text' in st.session_state:
        st.markdown("---")
        st.markdown(st.session_state['offline_video_blog_text'])
        st.markdown("---")
        
        custom_name_offline = st.text_input("Enter the Name of your Export File", placeholder="Export File Name", key="name_file_offline")
        base_name_off = custom_name_offline.strip() if custom_name_offline.strip() else "offline_video_summary"
        clean_filename_offline = f"{base_name_off}.txt"
        
        st.download_button(
            label="Export your Blog Summary",
            data=st.session_state['offline_video_blog_text'],
            file_name=clean_filename_offline,
            mime="text/plain",
            use_container_width=True, key="download_btn_offline", on_click="ignore"
        )